# KVO_01 — MOS_1_2 Musanga probability model and evaluation

Train a new Musanga crown/pixel model specifically for the fully georeferenced MOS_1_2 orthomosaic and its crown annotations, calibrate probabilities on a validation split, evaluate on an independent held-out test split, and predict a georeferenced Musanga probability raster for the full mosaic.

Outputs are written under `MyDrive/Colab Notebooks/Musanga/KVO_work/MOS_1_2/`.


In [ ]:
!pip -q install geopandas rasterio shapely pyogrio scikit-learn matplotlib tqdm segmentation-models-pytorch
from pathlib import Path
import os, json, math, random, warnings
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window, from_bounds
from rasterio.features import rasterize
from shapely.geometry import box
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss, precision_recall_curve, roc_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from google.colab import drive
warnings.filterwarnings('ignore')
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)


In [ ]:
DRIVE_ROOT=Path('/content/drive')
MYDRIVE=DRIVE_ROOT/'MyDrive'
if not MYDRIVE.exists() or not any(MYDRIVE.iterdir()): drive.mount(str(DRIVE_ROOT), force_remount=False)
PROJECT_ROOT=MYDRIVE/'Colab Notebooks'/'Musanga'
DATA_ROOT=PROJECT_ROOT/'data'/'Geo_Ref'
ANN_ROOT=DATA_ROOT/'Annotations'
OUT=PROJECT_ROOT/'KVO_work'/'MOS_1_2'
MODEL_DIR=OUT/'model'; EVAL_DIR=OUT/'evaluation'; PROB_DIR=OUT/'probability'; QA_DIR=OUT/'qa'; TRAIN_DIR=OUT/'training'
for p in [OUT,MODEL_DIR,EVAL_DIR,PROB_DIR,QA_DIR,TRAIN_DIR]: p.mkdir(parents=True,exist_ok=True)
MOSAIC=DATA_ROOT/'MOS_1_2_kvo.tif'
ANNOT=ANN_ROOT/'Tree_Annotations_MOS_1_2_kvo.shp'
print('MOSAIC exists:',MOSAIC.exists(), MOSAIC)
print('ANNOT exists:',ANNOT.exists(), ANNOT)
assert MOSAIC.exists(), 'Missing MOS_1_2_kvo.tif'
assert ANNOT.exists(), 'Missing Tree_Annotations_MOS_1_2_kvo.shp'


## 1 — Input audit and annotation field discovery


In [ ]:
with rasterio.open(MOSAIC) as src:
    raster_crs=src.crs; transform=src.transform; width=src.width; height=src.height; count=src.count; nodata=src.nodata
    bounds=src.bounds; gsd=(abs(src.transform.a),abs(src.transform.e)); dtype=src.dtypes[0]
print({'crs':str(raster_crs),'size':(width,height),'bands':count,'gsd':gsd,'dtype':dtype,'nodata':nodata,'bounds':tuple(bounds)})
ann=gpd.read_file(ANNOT)
print('annotation rows:',len(ann),'CRS:',ann.crs)
print('columns:',list(ann.columns))
print(ann.head())
if ann.crs!=raster_crs: ann=ann.to_crs(raster_crs)
ann=ann[ann.geometry.notna() & ~ann.geometry.is_empty].copy().reset_index(drop=True)
ann['crown_id']=np.arange(len(ann))
rb=box(*bounds)
inside=ann.intersects(rb)
print('annotations overlapping raster:',inside.sum(),'/',len(ann))
ann=ann[inside].copy().reset_index(drop=True)
# Candidate label fields; inspect printed unique values before accepting.
candidates=['Clss_ID','class_id','Class_ID','class','Class','species','Species','label','Label','musanga','Musanga','type','Type']
for c in candidates:
    if c in ann.columns:
        vals=ann[c].dropna().astype(str).value_counts().head(20)
        print('\nFIELD',c); print(vals)


### Set the class mapping once after inspecting the cell above
`LABEL_FIELD` should identify the annotation class. Set `MUSANGA_VALUES` to values meaning Musanga. Everything else becomes non-Musanga unless listed in `IGNORE_VALUES`.


In [ ]:
LABEL_FIELD = "Class_ID"
MUSANGA_CLASS = 1

# Keep valid labelled crowns only.
ann = ann[ann[LABEL_FIELD].notna()].copy()
ann["class_id_num"] = pd.to_numeric(ann[LABEL_FIELD], errors="coerce")
ann = ann[ann["class_id_num"].notna()].copy()
ann["class_id_num"] = ann["class_id_num"].astype(int)

# Binary target: Class_ID 1 = Musanga; all other labelled crown classes = non-Musanga.
ann["target"] = (ann["class_id_num"] == MUSANGA_CLASS).astype(np.uint8)

print("\nClass distribution:")
print(ann.groupby(["target", "class_id_num"]).size())
print(
    "\nMusanga crowns:", int(ann["target"].sum()),
    "\nOther crowns:", int((ann["target"] == 0).sum()),
    "\nTotal:", len(ann),
)
assert ann["target"].nunique() == 2, "Need both Musanga and non-Musanga crowns."


## 2 — Crown-level split and fixed-physical-size training chips


In [ ]:
PATCH_M = 20.0
TARGET_GSD_M = max(gsd)  # native MOS resolution; change only if desired
PATCH_PX = int(round(PATCH_M/TARGET_GSD_M)); PATCH_PX=max(128,min(512,PATCH_PX)); PATCH_PX=int(math.ceil(PATCH_PX/32)*32)
print('patch:',PATCH_M,'m ->',PATCH_PX,'px at approx',TARGET_GSD_M,'m GSD')
train_ids,temp_ids=train_test_split(np.arange(len(ann)),test_size=0.30,random_state=SEED,stratify=ann.target)
val_ids,test_ids=train_test_split(temp_ids,test_size=0.50,random_state=SEED,stratify=ann.iloc[temp_ids].target)
ann['split']='train'; ann.loc[val_ids,'split']='val'; ann.loc[test_ids,'split']='test'
print(ann.groupby(['split','target']).size())
ann.to_file(TRAIN_DIR/'MOS_1_2_crowns_split.gpkg',layer='crowns',driver='GPKG')


In [ ]:
# Fast I/O pattern reused from the original Musanga U-Net notebooks:
# copy the source mosaic once to Colab local SSD, build the cache locally, then train from RAM.
import shutil, time
torch.backends.cudnn.benchmark = True

LOCAL_SOURCE_DIR = Path("/content/musanga_training/sources")
LOCAL_SOURCE_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_MOSAIC = LOCAL_SOURCE_DIR / MOSAIC.name
if not LOCAL_MOSAIC.exists() or LOCAL_MOSAIC.stat().st_size != MOSAIC.stat().st_size:
    print(f"Copying mosaic once from Drive to local SSD ({MOSAIC.stat().st_size/1e9:.2f} GB)...")
    t0=time.time(); shutil.copy2(MOSAIC, LOCAL_MOSAIC)
    print(f"Local copy ready in {(time.time()-t0)/60:.1f} min")
else:
    print("Reusing local mosaic:", LOCAL_MOSAIC)

def read_rgb_chip(src, geom, size_px=PATCH_PX):
    c=geom.centroid; half=PATCH_M/2
    win=from_bounds(c.x-half,c.y-half,c.x+half,c.y+half,src.transform).round_offsets().round_lengths()
    arr=src.read([1,2,3],window=win,boundless=True,fill_value=0,
                 out_shape=(3,size_px,size_px),resampling=rasterio.enums.Resampling.bilinear)
    arr=np.moveaxis(arr,0,-1)
    if arr.dtype != np.uint8:
        a=arr.astype(np.float32); vv=a[a>0]
        lo,hi=np.percentile(vv,[1,99]) if len(vv) else (0,1)
        arr=np.clip((a-lo)/max(hi-lo,1e-6)*255,0,255).astype(np.uint8)
    return arr

# Only ~130 MB for 1,177 x 192 x 192 x RGB uint8: keep the complete dataset in RAM.
RAM_CACHE = Path("/content/kvo01_mos12_chips_uint8.npy")
RAM_IDS   = Path("/content/kvo01_mos12_crown_ids.npy")
if not RAM_CACHE.exists() or not RAM_IDS.exists():
    images=np.empty((len(ann),PATCH_PX,PATCH_PX,3),dtype=np.uint8)
    ids=np.empty(len(ann),dtype=np.int64)
    with rasterio.open(LOCAL_MOSAIC) as src:
        for j,row in enumerate(tqdm(ann.itertuples(index=False),total=len(ann),desc="Caching crowns to RAM")):
            images[j]=read_rgb_chip(src,row.geometry)
            ids[j]=int(row.crown_id)
    np.save(RAM_CACHE,images,allow_pickle=False); np.save(RAM_IDS,ids,allow_pickle=False)
else:
    images=np.load(RAM_CACHE,mmap_mode=None); ids=np.load(RAM_IDS)
id_to_pos={int(cid):i for i,cid in enumerate(ids)}
print(f"RAM cache ready: {images.nbytes/1e6:.1f} MB, {len(images)} crowns")

class CrownDataset(Dataset):
    def __init__(self,gdf,split,augment=False):
        self.g=gdf[gdf.split==split].reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.g)
    def __getitem__(self,i):
        row=self.g.iloc[i]; img=images[id_to_pos[int(row.crown_id)]]
        x=img.astype(np.float32)/255.0
        if self.augment:
            if np.random.rand()<0.5: x=x[:,::-1].copy()
            if np.random.rand()<0.5: x=x[::-1,:].copy()
            gamma=np.random.uniform(.8,1.2); x=np.clip(x,0,1)**gamma
            x=np.clip(x*np.random.uniform(.85,1.15),0,1)
        return torch.from_numpy(np.moveaxis(x,-1,0)).float(), torch.tensor(float(row.target)), int(row.crown_id)

train_ds=CrownDataset(ann,'train',True); val_ds=CrownDataset(ann,'val'); test_ds=CrownDataset(ann,'test')
# With data already in RAM, workers are unnecessary and can actually add overhead.
train_dl=DataLoader(train_ds,batch_size=128,shuffle=True,num_workers=0,pin_memory=True)
val_dl=DataLoader(val_ds,batch_size=256,shuffle=False,num_workers=0,pin_memory=True)
test_dl=DataLoader(test_ds,batch_size=256,shuffle=False,num_workers=0,pin_memory=True)
print("Train / val / test:",len(train_ds),len(val_ds),len(test_ds),"| batches/epoch:",len(train_dl))


## 3 — Train a new MOS-specific model

This notebook uses an ImageNet-pretrained ResNet18 encoder with a binary classifier head. The training unit is an independently digitized crown patch. This is intentionally crown-level rather than pixel-level because the MOS annotation dataset consists of crown labels. Probability evaluation is therefore statistically cleaner than treating millions of within-crown pixels as independent observations.


In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
model=resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc=nn.Linear(model.fc.in_features,1)
model=model.to(DEVICE)
pos=max(1,int(ann[ann.split=='train'].target.sum())); neg=max(1,int((1-ann[ann.split=='train'].target).sum()))
pos_weight=torch.tensor([neg/pos],device=DEVICE)
criterion=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='max',factor=.5,patience=2)

def predict_loader(dl):
    model.eval(); ys=[]; ps=[]; ids=[]
    with torch.no_grad():
        for x,y,cid in dl:
            p=torch.sigmoid(model(x.to(DEVICE)).squeeze(1)).cpu().numpy()
            ys.extend(y.numpy()); ps.extend(p); ids.extend(cid.numpy())
    return np.array(ys),np.array(ps),np.array(ids)

scaler=torch.amp.GradScaler("cuda",enabled=(DEVICE=="cuda"))
best=-np.inf; patience=6; wait=0; history=[]
for epoch in range(1,31):
    model.train(); losses=[]
    for x,y,_ in train_dl:
        x=x.to(DEVICE); y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda",enabled=(DEVICE=="cuda")):
            logits=model(x).squeeze(1); loss=criterion(logits,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); losses.append(loss.item())
    yv,pv,_=predict_loader(val_dl)
    auc=roc_auc_score(yv,pv); ap=average_precision_score(yv,pv)
    history.append({'epoch':epoch,'loss':np.mean(losses),'val_auc':auc,'val_ap':ap})
    print(f'epoch {epoch:02d} loss={np.mean(losses):.4f} val_AUC={auc:.4f} val_AP={ap:.4f}')
    sched.step(ap)
    if ap>best:
        best=ap; wait=0; torch.save(model.state_dict(),MODEL_DIR/'MOS_1_2_resnet18_best.pt')
    else:
        wait+=1
        if wait>=patience: break
pd.DataFrame(history).to_csv(EVAL_DIR/'training_history.csv',index=False)
model.load_state_dict(torch.load(MODEL_DIR/'MOS_1_2_resnet18_best.pt',map_location=DEVICE))


## 4 — Probability calibration on validation crowns; independent test evaluation


In [ ]:
yv,pv,idv=predict_loader(val_dl)
yt,pt,idt=predict_loader(test_dl)
# Compare Platt and isotonic on validation by in-sample Brier only; choose conservative Platt unless isotonic improves materially.
platt=LogisticRegression(solver='lbfgs').fit(pv.reshape(-1,1),yv.astype(int))
pv_platt=platt.predict_proba(pv.reshape(-1,1))[:,1]
iso=IsotonicRegression(out_of_bounds='clip').fit(pv,yv)
pv_iso=iso.predict(pv)
b_platt=brier_score_loss(yv,pv_platt); b_iso=brier_score_loss(yv,pv_iso)
calibrator='isotonic' if b_iso < b_platt*0.95 else 'platt'
print('validation Brier raw/platt/isotonic:',brier_score_loss(yv,pv),b_platt,b_iso,'selected:',calibrator)
if calibrator=='platt':
    calibrate=lambda p: platt.predict_proba(np.asarray(p).reshape(-1,1))[:,1]
else:
    calibrate=lambda p: iso.predict(np.asarray(p))
ptc=np.clip(calibrate(pt),1e-6,1-1e-6)
metrics={
 'n_test':int(len(yt)),'n_test_musanga':int(yt.sum()),'n_test_other':int(len(yt)-yt.sum()),
 'roc_auc_raw':float(roc_auc_score(yt,pt)),'average_precision_raw':float(average_precision_score(yt,pt)),
 'brier_raw':float(brier_score_loss(yt,pt)),'logloss_raw':float(log_loss(yt,np.clip(pt,1e-6,1-1e-6))),
 'roc_auc_calibrated':float(roc_auc_score(yt,ptc)),'average_precision_calibrated':float(average_precision_score(yt,ptc)),
 'brier_calibrated':float(brier_score_loss(yt,ptc)),'logloss_calibrated':float(log_loss(yt,ptc)),
 'calibrator':calibrator
}
print(json.dumps(metrics,indent=2))
(EVAL_DIR/'MOS_1_2_test_metrics.json').write_text(json.dumps(metrics,indent=2))
pred_df=pd.DataFrame({'crown_id':idt,'truth':yt.astype(int),'prob_raw':pt,'prob_calibrated':ptc})
pred_df.to_csv(EVAL_DIR/'MOS_1_2_test_crown_probabilities.csv',index=False)


In [ ]:
fig,ax=plt.subplots(figsize=(6,5))
fpr,tpr,_=roc_curve(yt,ptc); ax.plot(fpr,tpr,label=f"AUC={metrics['roc_auc_calibrated']:.3f}"); ax.plot([0,1],[0,1],'--'); ax.set(xlabel='False positive rate',ylabel='True positive rate',title='MOS_1_2 held-out crown ROC'); ax.legend(); fig.tight_layout(); fig.savefig(EVAL_DIR/'roc_test.png',dpi=180); plt.show()
fig,ax=plt.subplots(figsize=(6,5)); prec,rec,_=precision_recall_curve(yt,ptc); ax.plot(rec,prec,label=f"AP={metrics['average_precision_calibrated']:.3f}"); ax.set(xlabel='Recall',ylabel='Precision',title='MOS_1_2 held-out crown PR'); ax.legend(); fig.tight_layout(); fig.savefig(EVAL_DIR/'pr_test.png',dpi=180); plt.show()
fig,ax=plt.subplots(figsize=(6,5)); bins=np.linspace(0,1,11); ids=np.digitize(ptc,bins)-1; xs=[]; ys=[]; ns=[]
for b in range(10):
    m=ids==b
    if m.sum(): xs.append(ptc[m].mean()); ys.append(yt[m].mean()); ns.append(m.sum())
ax.plot([0,1],[0,1],'--'); ax.plot(xs,ys,'o-'); ax.set(xlabel='Mean calibrated probability',ylabel='Observed Musanga fraction',title='Reliability — held-out crowns'); fig.tight_layout(); fig.savefig(EVAL_DIR/'reliability_test.png',dpi=180); plt.show()


## 5 — Full-mosaic georeferenced probability inference

The crown classifier is applied in a sliding-window fashion. Each window predicts the probability that the window is centred on a Musanga crown. Predictions are accumulated and averaged into a continuous raster. This raster is a **Musanga crown-likeness probability surface**, not yet fractional canopy cover.


In [ ]:
# Efficient full-mosaic inference:
# 1) copy the large TIFF once from Drive to Colab local SSD;
# 2) read it sequentially in row strips;
# 3) batch many windows per GPU call;
# 4) write only the compact coarse probability grid locally;
# 5) stream-upsample rows to the final georeferenced uint16 GeoTIFF.
import shutil, time
# LOCAL_MOSAIC was already created before training; reuse it here.
print('Reusing local mosaic for inference:', LOCAL_MOSAIC)

STRIDE_PX=max(48,PATCH_PX//2)  # ~10 m centres for a 20 m patch; 4x fewer windows than previous code
BATCH=256 if DEVICE=='cuda' else 32
OUT_TIF=PROB_DIR/'MOS_1_2_musanga_probability_uint16.tif'

with rasterio.open(LOCAL_MOSAIC) as src:
    H,W=src.height,src.width
    rows=np.arange(0,H,STRIDE_PX,dtype=int); cols=np.arange(0,W,STRIDE_PX,dtype=int)
    prob_grid=np.full((len(rows),len(cols)),np.nan,np.float32)
    half=PATCH_PX//2
    model.eval()
    t0=time.time()

    for ri,r in enumerate(tqdm(rows,desc='Fast mosaic inference')):
        imgs=[]; positions=[]
        for ci,c in enumerate(cols):
            win=Window(c-half,r-half,PATCH_PX,PATCH_PX)
            arr=src.read([1,2,3],window=win,boundless=True,fill_value=0,
                         out_shape=(3,PATCH_PX,PATCH_PX),
                         resampling=rasterio.enums.Resampling.bilinear)
            if not np.any(arr): continue
            a=np.moveaxis(arr,0,-1)
            if a.dtype!=np.uint8:
                af=a.astype(np.float32); vv=af[af>0]
                lo,hi=np.percentile(vv,[1,99]) if len(vv) else (0,1)
                a=np.clip((af-lo)/(max(hi-lo,1e-6))*255,0,255).astype(np.uint8)
            imgs.append(np.moveaxis(a.astype(np.float32)/255.0,-1,0))
            positions.append(ci)
            if len(imgs)>=BATCH:
                x=torch.from_numpy(np.stack(imgs)).float().to(DEVICE,non_blocking=True)
                with torch.inference_mode():
                    p=torch.sigmoid(model(x).squeeze(1)).cpu().numpy()
                pc=calibrate(p)
                prob_grid[ri,np.asarray(positions)]=pc
                imgs=[]; positions=[]
        if imgs:
            x=torch.from_numpy(np.stack(imgs)).float().to(DEVICE,non_blocking=True)
            with torch.inference_mode():
                p=torch.sigmoid(model(x).squeeze(1)).cpu().numpy()
            prob_grid[ri,np.asarray(positions)]=calibrate(p)

    print(f"Inference grid {len(rows)} x {len(cols)} completed in {(time.time()-t0)/60:.1f} min")

    # Write the full-resolution georeferenced product block-by-block: no H x W float array in RAM.
    profile=src.profile.copy()
    profile.update(count=1,dtype='uint16',nodata=65535,compress='deflate',
                   tiled=True,predictor=2,BIGTIFF='YES')
    coarse_transform=src.transform*Affine.scale(STRIDE_PX,STRIDE_PX)
    with rasterio.open(OUT_TIF,'w',**profile) as dst:
        block_rows=1024
        for y0 in tqdm(range(0,H,block_rows),desc='Writing probability GeoTIFF'):
            hh=min(block_rows,H-y0)
            block=np.full((hh,W),np.nan,np.float32)
            dst_transform=src.window_transform(Window(0,y0,W,hh))
            reproject(prob_grid,block,
                      src_transform=coarse_transform,src_crs=src.crs,
                      dst_transform=dst_transform,dst_crs=src.crs,
                      resampling=rasterio.enums.Resampling.bilinear,
                      src_nodata=np.nan,dst_nodata=np.nan)
            enc=np.where(np.isfinite(block),
                         np.clip(np.round(block*10000),0,10000),65535).astype(np.uint16)
            dst.write(enc,1,window=Window(0,y0,W,hh))
print('Saved probability raster:',OUT_TIF)


In [ ]:
summary={'mosaic':str(MOSAIC),'annotations':str(ANNOT),'label_field':LABEL_FIELD,'musanga_class_id':int(MUSANGA_CLASS),'n_crowns':int(len(ann)),'n_train':int((ann.split=='train').sum()),'n_val':int((ann.split=='val').sum()),'n_test':int((ann.split=='test').sum()),'patch_m':PATCH_M,'patch_px':PATCH_PX,'stride_px':STRIDE_PX,'metrics':metrics,'probability_raster':str(OUT_TIF)}
(OUT/'KVO_01_MOS_1_2_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
print('\nDONE — outputs:',OUT)
